# 🔌 ROS2 커스텀 인터페이스 정의서 (Interface Specification)

> [!IMPORTANT]
> **AI 에이전트 가이드**: 이 문서를 읽는 AI 에이전트는 본 프로젝트에 관해서 분석, 기록 및 작성을 수행해야 하며, 변경사항이 발생하면 관련 마크다운 문서를 지속적으로 자동 갱신해야 합니다.

이 문서는 **`cobot3_interfaces`** 패키지 내 서비스(Service) 및 액션(Action) 정의 파일들의 최신 명세를 정리한 문서입니다. QR코드 시스템 도입과 작업대 8칸 확장(2026-06-04)으로 기존 ArUco `int32` ID 필드가 QR `string` ID 필드로 전면 교체되고, 8개 슬롯 범위 정보 및 우선순위 대기열 제어 상태가 인터페이스 명세와 통합 반영되었습니다.

---

## 📌 1. 인터페이스 변경 이력 요약

### v2.1 → v2.2 (AMR 플릿 하이브리드 연동 규격 반영, 2026-06-05)
* **`ManageWorkstation.action` Goal 확장**: AMR 이동 시 QR ID와 좌표의 동시 지정을 지원하기 위해 `target_qr_id`, `target_x`, `target_y`, `target_yaw` 필드를 추가했습니다.
* **상태 모니터링 JSON 토픽 추가**: 제어 채널과 분리된 실시간 상태 공유 채널 구축을 위해 `/fleet/amr_states`, `/fleet/workstation_states`, `/fleet/package_states`, `/fleet/task_events` 토픽 명세를 연동 문서 규격에 통합하였습니다.

### v2.0 → v2.1 (작업대 8칸 확장 및 우선순위 큐 동기화, 2026-06-04)
* **슬롯 범위 주석 정합성 동기화**: `ReportInboundProgress.srv` 및 `StartPackaging.action` 내의 슬롯 번호 가이드를 기존 `1~4`에서 실제 적용된 8칸 레이아웃 규격인 **`1~8`**로 수정하였습니다.
* **우선순위 제어 연동**: 우선순위 큐(Redis Sorted Set) 및 180도 회전 시퀀스(`ROTATE_WORKSTATION`) 도입에 따라 `ManageWorkstation.action`의 목적지 물리 위치 제어 상태(`_A_ROTATING`, `_A`, `_B` 등)와의 정합성을 보장했습니다.

### v1.0 → v2.0 (ArUco → QR 전환, 2026-06-04)
기존의 정수형 ArUco 마커 번호(`int32 aruco_id`)를 사용하던 구조에서 **문자열 QR코드 ID**(`string qr_id`)로 전면 전환하였습니다.

* **타입 변경**: `int32` → `string` (QR코드는 가변 길이 문자열을 인코딩하므로)
* **필드 이름 변경**: `aruco_id` → `qr_id`, `workstation_aruco_id` → `workstation_qr_id`, `package_aruco_id` → `package_qr_id`
* **호환성**: ArUco 기반 레거시 코드는 `qr_id` 필드에 빈 문자열(`""`)을 전송하고, 기존 `package_id` / `workstation_id` 문자열 필드를 사용하면 됩니다.
* **마커 우선 처리**: 관제탑 노드는 `qr_id` 값이 비어있지 않을 경우(`!= ""`) 해당 QR 정보로 DB에서 데이터를 우선적으로 조회 및 매핑합니다.

---

## 🛠️ 2. ROS2 서비스(Service) 정의 상세

### ① `GetPackageRoute.srv`
컨베이어 분류기(`bg2`)가 입고 상자를 스캔한 뒤 목적지 날짜를 조회할 때 사용됩니다.

**파일 경로**: `src/cobot3_interfaces/srv/GetPackageRoute.srv`

| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Request** | `package_id` | `string` | 택배 상자 바코드 ID (기본값: 공백 허용) |
| | `customer_name` | `string` | 수령인 성함 (기본값: 공백 허용) |
| | `qr_id` | `string` | 상자에 부착된 QR코드 ID (예: `"PKG_RAND_001"`) |
| **Response** | `route_destination` | `string` | 분류 목적지 배송예정 날짜 (예: `"2026-06-01"`) |

---

### ② `CheckWarehouseStatus.srv`
적재 매니퓰레이터(`sg2_in_XX`)가 적재를 수행하기 전, 해당 패키지가 이미 작업대 또는 창고에 적재되어 보관 중인지 `package_id`를 기준으로 중복 검사합니다. (기존의 단순 수령인 이름(`customer_name`) 기반 매핑에서 발생하던 완료된 과거 이력 오인 현상을 수정하여 패키지 ID 단위로 정밀하게 검증합니다.)

**파일 경로**: `src/cobot3_interfaces/srv/CheckWarehouseStatus.srv`


| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Request** | `customer_name` | `string` | 검사 대상 수령인 성함 (기본값: 공백 허용) |
| | `package_id` | `string` | 검사 대상 상자 ID (기본값: 공백 허용) |
| | `qr_id` | `string` | 검사 대상 상자의 QR코드 ID (예: `"PKG_RAND_001"`) |
| **Response** | `is_already_in_warehouse` | `bool` | 기존 동일 물품 존재 여부 (`true` / `false`) |

---

### ③ `ReportInboundProgress.srv`
적재 매니퓰레이터(`sg2_in_XX`)가 2x4 작업대에 상자를 안전하게 얹어놓을 때마다 관제 센터로 진척도를 실시간 보고합니다.

**파일 경로**: `src/cobot3_interfaces/srv/ReportInboundProgress.srv`

| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Request** | `workstation_id` | `string` | 작업대 고유 ID (기본값: 공백 허용) |
| | `robot_id` | `string` | 적재를 보고하는 로봇 식별자 (예: `"sg2_in_01"`) |
| | `filled_slots_count` | `int32` | 적재된 슬롯 위치 번호 (`1` ~ `8`) |
| | `package_id` | `string` | 적재한 패키지 ID (기본값: 공백 허용) |
| | `workstation_qr_id` | `string` | 적재 중인 작업대의 QR코드 ID (예: `"WORKSTATION_WS01"`) |
| | `package_qr_id` | `string` | 적재된 상자의 QR코드 ID (예: `"PKG_RAND_001"`) |
| **Response** | `success` | `bool` | 처리 결과 및 DB 반영 완료 여부 (`true` / `false`) |

---

## 🏃 3. ROS2 액션(Action) 정의 상세

### ① `MovePackage.action`
AMR(자율 이송 로봇)에게 창고 직송 등의 단일 패키지 강제 이송을 명령할 때 사용됩니다.

**파일 경로**: `src/cobot3_interfaces/action/MovePackage.action`

| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Goal** | `package_id` | `string` | 이송 대상 상자 ID |
| | `customer_name` | `string` | 수령인 성함 |
| | `destination_zone` | `string` | 창고 내 보관 구역 (예: `"ZONE_A"`) |
| | `package_qr_id` | `string` | 상자 고유 QR코드 ID (예: `"PKG_RAND_001"`) |
| **Result** | `success` | `bool` | 최종 이송 완료 여부 |
| | `error_msg` | `string` | 실패 시 에러 메시지 |
| **Feedback** | `current_position` | `string` | AMR의 현재 위치 좌표 또는 구역명 |
| | `progress` | `float32` | 이동 진행률 (0.0 ~ 100.0 %) |

---

### ② `ManageWorkstation.action`
AMR에게 작업대를 다른 공정존이나 보관용 창고로 이송하도록 지시합니다. (예: 완충 시 포장존 이송, 포장 완료 시 빈 작업대 회수 등)

**파일 경로**: `src/cobot3_interfaces/action/ManageWorkstation.action`

| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Goal** | `workstation_id` | `string` | 제어 대상 작업대 ID |
| | `start_location` | `string` | 출발 물리 위치 (예: `"sg2_in_01"`) |
| | `target_location` | `string` | 도착 물리 위치 (예: `"sg2_out_00_A"`) |
| | `workstation_qr_id` | `string` | 작업대 고유 QR코드 ID (예: `"WORKSTATION_WS01"`) |
| | `target_qr_id` | `string` | 목적지 바닥 QR ID (예: `"QR_0030"`) |
| | `target_x` | `float64` | 목적지 X 물리 좌표 (m) |
| | `target_y` | `float64` | 목적지 Y 물리 좌표 (m) |
| | `target_yaw` | `float64` | 목적지 Yaw 물리 각도 (rad) |
| **Result** | `success` | `bool` | 이송 완료 성공 여부 |
| **Feedback** | `distance_remaining` | `float32` | 목적지까지 남은 거리 (m) |
| | `status` | `string` | 현재 상태 (`"PICKING"`, `"NAVIGATING"`, `"PLACING"`) |

---

### ③ `StartPackaging.action`
포장 로봇(`sg2_out_00_A`)에게 특정 작업대에 도달한 상자 8칸의 포장 공정을 명령합니다.

**파일 경로**: `src/cobot3_interfaces/action/StartPackaging.action`

| 구분 | 필드명 (Field) | 데이터 타입 (Type) | 역할 (Role) |
| :--- | :--- | :--- | :--- |
| **Goal** | `workstation_id` | `string` | 포장 대상 작업대 ID |
| | `today_date` | `string` | 출고 ID 생성용 오늘 날짜 (YYYYMMDD) |
| | `workstation_qr_id` | `string` | 작업대 고유 QR코드 ID (예: `"WORKSTATION_WS01"`) |
| **Result** | `success` | `bool` | 포장 공정 완료 여부 |
| | `final_output_ids` | `string[]` | 생성된 고유 출고 ID 리스트 (포장 로봇 ID prefix 포함) |
| **Feedback** | `completed_slots` | `int32` | 현재 포장 완료된 슬롯 누적 갯수 (1~8) |
| | `last_packed_slot` | `string` | 직전에 포장 완료된 슬롯 번호 (예: `"slot_3"`) |

---

## 📊 4. Fleet 상태 모니터링 JSON 토픽 상세 (State Plane)

하이브리드 통신 아키텍처에 따라, 제어 명령(Action/Service) 채널과 분리되어 실시간 상태 모니터링 및 대시보드 연동을 위해 발행되는 JSON 형식의 ROS2 토픽 명세입니다. (메시지 유형: `std_msgs/msg/String` 내 직렬화된 JSON 문자열)

### ① `/fleet/amr_states`
* **발행 주기**: 1Hz 주기 (Periodic)
* **내용**: 각 AMR(자율 이송 로봇)의 현재 운전 상태, 위치(바닥 QR), 목표 위치, 배터리 잔량 및 가용 여부입니다.
* **JSON 페이로드 구조**:
  ```json
  {
    "AMR_01": {
      "state": "IDLE",
      "current_qr_id": "QR_0030",
      "target_qr_id": "",
      "carrying_workstation_id": null,
      "battery": 82.5,
      "available": true
    }
  }
  ```

### ② `/fleet/workstation_states`
* **발행 주기**: 1Hz 주기 (Periodic)
* **내용**: 물류창고 내 전체 작업대의 현재 물리적 위치(바닥 QR ID), 작업 상태, 적재 슬롯 상태(PostgreSQL packages 테이블 연계 동적 생성)입니다.
* **JSON 페이로드 구조**:
  ```json
  {
    "workstations": [
      {
        "workstation_id": "WS01",
        "workstation_qr_id": "WORKSTATION_WS01",
        "current_location": "QR_0030",
        "status": "WAITING",
        "slot_count": 8,
        "filled_slots": [1, 2, 3, 4],
        "reserved_by": null
      }
    ]
  }
  ```

### ③ `/fleet/package_states`
* **발행 주기**: 1Hz 주기 (Periodic)
* **내용**: 창고 내에 존재하는 활성 패키지들(상태가 `COMPLETED`가 아닌 패키지)의 실시간 상태 데이터입니다.
* **JSON 페이로드 구조**:
  ```json
  {
    "packages": [
      {
        "package_id": "PKG_RAND_001",
        "customer_name": "김태희",
        "route_zone": "2026-06-01",
        "status": "WAITING",
        "outbound_id": null,
        "workstation_id": null,
        "slot_number": null,
        "qr_id": "PKG_RAND_001"
      }
    ]
  }
  ```

### ④ `/fleet/task_events`
* **발행 주기**: 이벤트 발생 시 즉시 발행 (Event-driven)
* **내용**: 관제 센터가 생성하고 AMR에 할당하여 실행 완료 또는 실패에 이르는 태스크 생애주기 전반의 상태 변경 로그입니다.
* **JSON 페이로드 구조**:
  ```json
  {
    "schema_version": "1.0",
    "timestamp": 1780626168.9948,
    "task_id": "uuid-string",
    "type": "MOVE_WORKSTATION",
    "priority": 80,
    "workstation_id": "WS01",
    "workstation_qr_id": "WORKSTATION_WS01",
    "start_location": "spot_01",
    "target_location": "sg2_in_01_A",
    "status": "ASSIGNED",
    "assigned_amr": "AMR_01"
  }
  ```

## 📅 4. 동적 출고일(route_zone) 및 실제 날짜 기준 영업일 전환 / 이월 적재 규칙

기존의 고정 날짜 처리 방식에서 탈피하여, 실제 달력 날짜 기준으로 영업일을 관리하고 이월된 작업대에 패키지를 이어서 적재하며, 3개 입고 라인을 고정식으로 운영하는 아키텍처를 적용했습니다.

### ① 동적 날짜 매핑 및 라인 고정 규칙
1. **라인 역할 고정**:
   * **1번 라인 (`sg2_in_01`)**: 무조건 **오늘 (Today)** 날짜 물량 적재
   * **2번 라인 (`sg2_in_02`)**: 무조건 **내일 (Tomorrow)** 날짜 물량 적재
   * **3번 라인 (`sg2_in_03`)**: 무조건 **모레 (Day After)** 날짜 물량 적재
2. **날짜 동적 승격**: 오늘 물량이 모두 출고 완료되면 시스템 상태가 `PENDING_TRANSITION`으로 전환되며, 명시적인 전환 API 호출 시 영업일이 변경됩니다.

### ② 실제 영업일 전환 API 및 이월 적재 (Carry-over)
* **영업일 전환 API (`/api/start_next_day`)**:
  * 호출 시, Redis에 기록된 `system:today_date`가 실제 하루 뒤 날짜로 업데이트되고 `system:day_status`는 `RUNNING` 상태로 원복됩니다.
  * 물리적인 작업대 이송(Line Shift) 태스크가 `queue:amr_tasks`에 발행됩니다:
    * 2번 라인(`sg2_in_02`)에 대기 중이던 작업대(내일 물량 실림) ➡️ 1번 라인(`sg2_in_01`)으로 이동
    * 3번 라인(`sg2_in_03`)에 대기 중이던 작업대(모레 물량 실림) ➡️ 2번 라인(`sg2_in_02`)으로 이동
    * 3번 라인에는 창고에서 새로운 빈 작업대가 공급됩니다.
  * 전환 시 Redis의 `system:inbound_started` 플래그는 `'false'`로 리셋됩니다.
* **조기 포장 방지 플래그 (`system:inbound_started`)**:
  * 영업일 전환 직후, 아직 당일의 신규 패키지 CSV 파일이 업로드되지 않은 상황에서 이월된 작업대(예: 6개 적재 상태)가 포장존으로 자동 공급(플러시)되는 것을 막기 위해 도입되었습니다.
  * 대시보드를 통해 당일 패키지 CSV 파일이 성공적으로 업로드(`/api/upload_packages`)되면 이 플래그가 `'true'`로 설정되어 본격적인 적재 및 포장 자동 공급 루프가 활성화됩니다.
  * 1번 라인으로 이동한 이월 작업대는 신규 패키지가 입고되어 나머지 슬롯(Slot 7, Slot 8)이 마저 채워질 때까지 안정적으로 대기한 뒤 완충 시 포장존으로 이송됩니다.
